In [ ]:
import pandas as pd
import numpy as np

from haversine import haversine_vector, Unit

In [ ]:
from haversine import haversine_vector, Unit
import numpy as np

# 一個起始點
start_point = (30.6574, 104.0658)  # 成都

# 多個目的地
destinations = [
    (29.5630, 106.5516), # 重慶
    (26.5878, 106.7072), # 貴陽
    (34.2648, 108.9441)  # 西安
]

# 使用 haversine_vector 計算距離矩陣
# 這會回傳一個 numpy 陣列，包含 start_point 到每一個 destination 的距離
distances_km = haversine_vector([start_point], destinations, Unit.KILOMETERS, comb=True)

print("起始點：", start_point)
print("目的地列表：", destinations)
print("\\n計算出的距離矩陣 (公里):")
print(distances_km)

# 如果您有兩組點，也可以計算它們之間的配對距離
# 使用字典來儲存地點，更具可讀性
points_A_dict = {
    "成都": (30.6574, 104.0658),
    "上海": (31.2304, 121.4737)
}

points_B_dict = {
    "重慶": (29.5630, 106.5516),
    "貴陽": (26.5878, 106.7072),
    "西安": (34.2648, 108.9441)
}

# 從字典中提取座標列表和名稱列表
points_A_coords = list(points_A_dict.values())
points_A_names = list(points_A_dict.keys())

points_B_coords = list(points_B_dict.values())
points_B_names = list(points_B_dict.keys())

# 計算距離矩陣
distances_matrix = haversine_vector(points_A_coords, points_B_coords, Unit.KILOMETERS, comb=True)

# 建立 DataFrame，並使用字典的鍵作為 index 和 columns
distances_matrix_df = pd.DataFrame(distances_matrix, index=points_B_names, columns=points_A_names).round(2)
distances_matrix_df

In [ ]:
cd_more_one_df = pd.read_csv("成都_strategy_A_multiple_stations_per_cpo.csv")
cd_emo_df = pd.read_csv("成都_public_competitor_stations.csv")
cd_emo_df.rename(columns={"name": "station_name"}, inplace=True)
cd_emo_df = cd_emo_df[cd_emo_df["cpo_final"] != "蔚来"].reset_index(drop=True)

In [ ]:
cd_more_one_df[["station_name", "longitude", "latitude"]]
cd_emo_df[["station_name", "longitude", "latitude"]]

In [ ]:
# 1. 從 DataFrame 中提取座標和名稱
points_A_coords = list(zip(cd_more_one_df['latitude'], cd_more_one_df['longitude']))
points_A_names = cd_more_one_df['station_name'].tolist()

points_B_coords = list(zip(cd_emo_df['latitude'], cd_emo_df['longitude']))
points_B_names = cd_emo_df['station_name'].tolist()

# 2. 計算距離矩陣
# haversine_vector 返回的矩陣形狀為 (len(points_A), len(points_B))
distance_matrix_ab = haversine_vector(points_A_coords, points_B_coords, Unit.KILOMETERS, comb=True)

# 3. 建立 DataFrame
# index 應對應 points_A, columns 應對應 points_B
distance_df = pd.DataFrame(distance_matrix_ab, index=points_B_names, columns=points_A_names).round(2)

distance_df.min(), distance_df.idxmin()

In [ ]:
# 筛选出包含“理想”的行
li_auto_df = distance_df[distance_df.index.str.contains('理想')]

# 筛选出包含“小鹏”的行
xpeng_df = distance_df[distance_df.index.str.contains('小鹏')]

# 找到每个原始站点到“理想”品牌的最近距离
closest_li_auto = li_auto_df.min()

# 找到每个原始站点到“小鹏”品牌的最近距离
closest_xpeng = xpeng_df.min()

print("--- 最近的理想站点距离 ---")
print(closest_li_auto.sort_values().head())

print("\n--- 最近的小鹏站点距离 ---")
print(closest_xpeng.sort_values().head())

In [ ]:
# --- 执行替换操作 ---

# 1. 找出要替换的原始站点名称
station_to_replace_for_li = '愉秒充新园大道充电站'
station_to_replace_for_xpeng = '昊铂成都王府井百货总府路店超充站'

# 2. 找到对应的最佳OME站点名称
# idxmin() 会找到每列（每个原始站点）的最小值对应的索引（OME站点名）
li_replacement_name = li_auto_df.idxmin()[station_to_replace_for_li]
xpeng_replacement_name = xpeng_df.idxmin()[station_to_replace_for_xpeng]

print(f"'{station_to_replace_for_li}' 将被替换为理想站点: '{li_replacement_name}'")
print(f"'{station_to_replace_for_xpeng}' 将被替换为小鹏站点: '{xpeng_replacement_name}'")

# 3. 从OME数据表 (cd_emo_df) 中获取这两个新站点的完整信息
li_station_data = cd_emo_df[cd_emo_df['station_name'] == li_replacement_name]
xpeng_station_data = cd_emo_df[cd_emo_df['station_name'] == xpeng_replacement_name]

# 4. 创建最终的目标列表副本
final_target_df = cd_more_one_df.copy()

# 5. 删除旧的站点
final_target_df = final_target_df[~final_target_df['station_name'].isin([station_to_replace_for_li, station_to_replace_for_xpeng])]

# 6. 合并新站点
# 注意：我们需要确保列名一致，cd_emo_df 可能缺少一些列，合并时会自动用 NaN 填充
final_target_df = pd.concat([final_target_df, li_station_data, xpeng_station_data], ignore_index=True)

# 7. 显示并保存最终名单
print("\n--- 更新后的目标清单 ---")
display(final_target_df)

# 保存到新的CSV文件
final_target_df.to_csv("final_target_list_CD.csv", index=False, encoding='utf-8-sig')
print("\n最终名单已保存到 'final_target_list_CD.csv'")

In [ ]:
# --- 在路线计划中直接替换并更新坐标 ---

# 1. 建立明确的替换映射
replacement_map = {
    station_to_replace_for_li: li_replacement_name,
    station_to_replace_for_xpeng: xpeng_replacement_name
}

# 2. 建立新站点的坐标映射
coords_map = {
    li_replacement_name: (li_station_data.iloc[0]['latitude'], li_station_data.iloc[0]['longitude']),
    xpeng_replacement_name: (xpeng_station_data.iloc[0]['latitude'], xpeng_station_data.iloc[0]['longitude'])
}

print("--- 站点替换关系 ---")
print(replacement_map)
print("\n--- 新站点坐标 ---")
print(coords_map)

# 3. 加载原始路线文件
route_df = pd.read_csv("output/report_CD_enriched.csv")

# 4. 循环执行替换
for old_name, new_name in replacement_map.items():
    # 获取新站点的坐标
    new_lat, new_lon = coords_map[new_name]
    
    # 找到所有需要替换的行
    rows_to_update = route_df['目的地'] == old_name
    
    # 替换名称
    route_df.loc[rows_to_update, '目的地'] = new_name
    
    # 更新坐标
    route_df.loc[rows_to_update, '目的地緯度'] = new_lat
    route_df.loc[rows_to_update, '目的地經度'] = new_lon

# 5. 显示并保存更新后的路线
print("\n--- 更新后的路线计划 (部分预览) ---")
# 筛选出被修改的行和它们前后的记录，以供查验
display(route_df[route_df['目的地'].isin(replacement_map.values())])

route_df.to_csv("output/report_CD_replaced.csv", index=False, encoding='utf-8-sig')
print("\n已将更新后的完整路线保存到 'output/report_CD_replaced.csv'")

## CPO 統計

In [ ]:
import pandas as pd

# Load the station data for Chengdu
cd_stations_df = pd.read_csv('output/all_map_stations_CD.csv')

# Get the unique operator names
unique_cpos = cd_stations_df['operator_name'].unique()

# Print the list of unique CPOs
print("成都地区所有不重复的CPO列表:")
for cpo in sorted(unique_cpos):
    print(f"- {cpo}")

print(f"\n总计: {len(unique_cpos)} 个不重复的CPO")

In [ ]:
pd.DataFrame(unique_cpos)

In [ ]:
import pandas as pd
import glob

# 使用 glob 找到 Day/day1 和 Day/day2 下的所有 mission_test_records.csv
log_files = glob.glob('Day/day*/mission_test_records.csv')

# 檢查是否找到了檔案
if not log_files:
    print("錯誤：在 'Day/day1/' 或 'Day/day2/' 目錄下沒有找到 'mission_test_records.csv' 檔案。")
else:
    # 讀取所有日誌檔案並將它們合併成一個 DataFrame
    all_logs_df = pd.concat((pd.read_csv(f) for f in log_files), ignore_index=True)

    # 從 'CPO Name' 欄位中獲取不重複的 CPO 名稱
    # 使用 .dropna() 來忽略可能存在的空值
    tested_cpos = all_logs_df['CPO Name'].dropna().unique()

    # 打印出已測試的 CPO 列表
    print("Day 1 & Day 2 已測試的所有不重複的CPO列表:")
    for cpo in sorted(tested_cpos):
        print(f"- {cpo}")

    print(f"\n總計: {len(tested_cpos)} 個不重複的CPO")

In [ ]:
import pandas as pd
import glob
import numpy as np

# --- 步骤 1: 获取成都地区所有的 CPO 列表 ---
all_stations_df = pd.read_csv('output/all_map_stations_CD.csv')
all_cpos_series = all_stations_df['operator_name'].dropna().unique()
# 排除 'test' CPO (不区分大小写)
all_cpos_list = [cpo for cpo in all_cpos_series if 'test' not in cpo.lower()]


# --- 步骤 2: 获取 Day 1 & Day 2 已测试过的 CPO 列表 ---
log_files = glob.glob('Day/day*/mission_test_records.csv')

if not log_files:
    print("错误：在 'Day/day*/' 目录下没有找到 'mission_test_records.csv' 檔案。")
    tested_cpos_list = []
else:
    all_logs_df = pd.concat((pd.read_csv(f) for f in log_files), ignore_index=True)
    tested_cpos_series = all_logs_df['CPO Name'].dropna().unique()
    # 排除 'test' CPO (不区分大小写)
    tested_cpos_list = [cpo for cpo in tested_cpos_series if 'test' not in cpo.lower()]


# --- 步骤 3: 创建对比表格 ---
# 创建一个 DataFrame 来进行对比
comparison_df = pd.DataFrame(sorted(all_cpos_list), columns=['CPO Name'])

# 标记已访问的 CPO
comparison_df['Visited'] = np.where(comparison_df['CPO Name'].isin(tested_cpos_list), '✅', '❌')


# --- 步骤 4: 显示结果 ---
print("成都地区 CPO 任务完成情况一览表:\n")
# 使用 to_string() 来确保所有行都被打印出来
print(comparison_df.to_string())

# 分别打印已完成和未完成的列表
print("\n" + "="*40)
print("\n✅ 已完成的 CPO 列表:")
visited_df = comparison_df[comparison_df['Visited'] == '✅']
for cpo in visited_df['CPO Name']:
    print(f"- {cpo}")
print(f"\n已完成数量: {len(visited_df)}")


print("\n" + "="*40)
print("\n❌ 待完成的 CPO 列表:")
not_visited_df = comparison_df[comparison_df['Visited'] == '❌']
for cpo in not_visited_df['CPO Name']:
    print(f"- {cpo}")
print(f"\n未完成数量: {len(not_visited_df)}")

In [ ]:
import pandas as pd
import glob
import numpy as np
from pypinyin import pinyin, Style

# --- 准备工作：定义一个函数，将汉字字符串转换为无声调的连续拼音 ---
def to_pinyin_flat(text):
    """将 '太能衝' 转换为 'tainengchong'"""
    # style=Style.NORMAL 表示无声调
    pinyin_list = pinyin(text, style=Style.NORMAL)
    # 将 [['tai'], ['neng'], ['chong']] 转换为 'tainengchong'
    return ''.join([item[0] for item in pinyin_list])

# --- 步骤 1: 获取成都地区所有的 CPO 列表 ---
all_stations_df = pd.read_csv('output/all_map_stations_CD.csv')
all_cpos_series = all_stations_df['operator_name'].dropna().unique()
all_cpos_list_orig = [cpo for cpo in all_cpos_series if 'test' not in str(cpo).lower()]
# 创建一个拼音版的列表用于比对
all_cpos_list_pinyin = [to_pinyin_flat(cpo) for cpo in all_cpos_list_orig]

# --- 步骤 2: 获取 Day 1 & Day 2 已测试过的 CPO 列表 ---
log_files = glob.glob('Day/day*/mission_test_records.csv')

if not log_files:
    print("错误：在 'Day/day*/' 目录下没有找到 'mission_test_records.csv' 檔案。")
    tested_cpos_list_pinyin = []
else:
    all_logs_df = pd.concat((pd.read_csv(f) for f in log_files), ignore_index=True)
    tested_cpos_series = all_logs_df['CPO Name'].dropna().unique()
    tested_cpos_list_orig = [cpo for cpo in tested_cpos_series if 'test' not in str(cpo).lower()]
    # 创建一个拼音版的列表用于比对
    tested_cpos_list_pinyin = [to_pinyin_flat(cpo) for cpo in tested_cpos_list_orig]

# --- 步骤 3: 创建对比表格 ---
# 使用原始名称创建 DataFrame
comparison_df = pd.DataFrame(sorted(all_cpos_list_orig), columns=['CPO Name'])
# 创建一个临时的拼音列用于比对
comparison_df['CPO Name Pinyin'] = comparison_df['CPO Name'].apply(to_pinyin_flat)

# 使用拼音列表进行 'isin' 判断
comparison_df['Visited'] = np.where(comparison_df['CPO Name Pinyin'].isin(tested_cpos_list_pinyin), '✅', '❌')

# 删除临时的拼音列
comparison_df.drop(columns=['CPO Name Pinyin'], inplace=True)

# --- 步骤 4: 显示结果 ---
print("成都地区 CPO 任务完成情况一览表 (已处理拼音差异):\n")
print(comparison_df.to_string())

# --- 步骤 5: 打印去重后的已完成和未完成列表 ---
# 使用集合操作来获取准确的已完成和未完成的拼音CPO列表
all_cpos_set_pinyin = set(all_cpos_list_pinyin)
tested_cpos_set_pinyin = set(tested_cpos_list_pinyin)

visited_pinyin = sorted(list(tested_cpos_set_pinyin))
not_visited_pinyin = sorted(list(all_cpos_set_pinyin - tested_cpos_set_pinyin))

# 为了显示更友好的原始名称，我们从拼音反向查找一个原始名称
pinyin_to_orig_map_all = {to_pinyin_flat(name): name for name in reversed(all_cpos_list_orig)}
pinyin_to_orig_map_tested = {to_pinyin_flat(name): name for name in reversed(tested_cpos_list_orig)}


print("\n" + "="*40)
print("\n✅ 已完成的 CPO 列表 (按拼音去重):")
for p in visited_pinyin:
    # 优先从测试过的列表里找原始名，找不到再从总列表里找
    display_name = pinyin_to_orig_map_tested.get(p, pinyin_to_orig_map_all.get(p, p))
    print(f"- {display_name}")
print(f"\n已完成数量: {len(visited_pinyin)}")

print("\n" + "="*40)
print("\n❌ 待完成的 CPO 列表 (按拼音去重):")
for p in not_visited_pinyin:
    display_name = pinyin_to_orig_map_all.get(p, p)
    print(f"- {display_name}")
print(f"\n未完成数量: {len(not_visited_pinyin)}")

In [ ]:
# 1. 定义在成都已经完成测试的 CPO 列表 (根据您的输入)
chengdu_completed_cpo = [
    "小桔充电", "逸安启", "蔚景云", "云快充", "星星充电", "特来电", 
    "中装天如", "太能充", "太空充电", "壳牌充电", "极氪能源", "达克云", 
    "港投中油", "国家电网", "广汽能源", "蔚来", "百源智联", "蜀道新能源", 
    "南方電網", "新電徒", "昊柏", "领充创享", "亿谦电子", "特斯拉", 
    "億天宸", "公交新能源", "城頭能源", "小鵬", "崑崙網電", "惠知電", 
    "万净极速充", "道畅充充", "理想超充", "武候啇程", "開麥斯", "嵐圖", 
    "快鰻", "四川明星新能源"
]

# 假设 cq_cpo_list 已经从上一个单元格成功加载
# 为了代码的独立性，我们再获取一次
try:
    cq_stations_df = pd.read_csv('output/all_map_stations_CQ.csv')
    cq_cpo_list = cq_stations_df['operator_name'].unique()
except NameError:
    print("错误：请先运行上一个单元格来加载 'cq_cpo_list'。")
    # 或者在这里重新加载
    # cq_stations_df = pd.read_csv('output/all_map_stations_CQ.csv')
    # cq_cpo_list = cq_stations_df['cpo'].unique()


# 2. 转换为集合以方便计算
chengdu_set = set(chengdu_completed_cpo)
chongqing_set = set(cq_cpo_list)

# 3. 计算交集和差集
already_tested_in_cq = chongqing_set.intersection(chengdu_set)
new_targets_in_cq = chongqing_set.difference(chengdu_set)

# 4. 打印结果
print("--- 重庆 CPO 测试计划 ---")
print(f"总计: {len(chongqing_set)} 个 CPO")
print(f"已在成都测试过: {len(already_tested_in_cq)} 个")
print(f"新的测试目标: {len(new_targets_in_cq)} 个")
print("\\n" + "="*30 + "\\n")

print("✅ 新的测试目标列表 (重庆):")
if not new_targets_in_cq:
    print("太棒了！所有重庆的CPO似乎都已经在成都测试过。")
else:
    for cpo in sorted(list(new_targets_in_cq)):
        print(f"- {cpo}")

print("\\n" + "="*30 + "\\n")

print("📋 已在成都测试过的 CPO 列表 (重庆也有):")
if not already_tested_in_cq:
    print("无。")
else:
    for cpo in sorted(list(already_tested_in_cq)):
        print(f"- {cpo}")


In [ ]:
import pandas as pd
from pypinyin import pinyin, Style

# --- 准备工作 ---
def to_pinyin_flat(text):
    """将 '太能衝' 转换为 'tainengchong'"""
    if not isinstance(text, str):
        return ""
    pinyin_list = pinyin(text, style=Style.NORMAL)
    return ''.join([item[0] for item in pinyin_list])

# --- 1. 定义 CPO 列表 ---

# 在成都已经完成测试的 CPO 列表
chengdu_completed_cpo = [
    "小桔充电", "逸安启", "蔚景云", "云快充", "星星充电", "特来电", 
    "中装天如", "太能充", "太空充电", "壳牌充电", "极氪能源", "达克云", 
    "港投中油", "国家电网", "广汽能源", "蔚来", "百源智联", "蜀道新能源", 
    "南方電網", "新電徒", "昊柏", "领充创享", "亿谦电子", "特斯拉", 
    "億天宸", "公交新能源", "城頭能源", "小鵬", "崑崙網電", "惠知電", 
    "万净极速充", "道畅充充", "理想超充", "武候啇程", "開麥斯", "嵐圖", 
    "快鰻", "四川明星新能源"
]

# 重庆的 CPO 列表
chongqing_cpo_list = [
    "小桔充电", "广汽能源", "智道", "星星充电", "润诚达", "逸安启", 
    "幸福千万家", "爱众云鲲", "百源智联", "云快充", "极氪能源", "蔚景云", 
    "金弦新能源", "驿满充电", "滨南时代", "重庆交运", "愉秒充", "特瓦特", 
    "能银链", "重庆中盾", "重庆能投", "朗新新能源", "重庆壳牌", 
    "重庆小马爱冲", "易安桩", "国家电网", "成都亿嘉电", "易安充", "渝快充", 
    "达克云", "开鑫充", "开鑫充电", "特来电", "智泊惠", "一码YiMa", 
    "南网电动", "库仑", "吉智能源", "小飞象"
]

# --- 2. 转换为拼音集合 ---
chengdu_pinyin_set = {to_pinyin_flat(cpo) for cpo in chengdu_completed_cpo}
chongqing_pinyin_set = {to_pinyin_flat(cpo) for cpo in chongqing_cpo_list}

# --- 3. 创建从拼音到原始名称的映射 ---
# 这有助于我们显示用户友好的中文名称
pinyin_to_orig_map_cq = {to_pinyin_flat(name): name for name in reversed(chongqing_cpo_list)}

# --- 4. 计算差集 (新的测试目标) ---
new_targets_pinyin = chongqing_pinyin_set - chengdu_pinyin_set
already_tested_pinyin = chongqing_pinyin_set.intersection(chengdu_pinyin_set)

# --- 5. 打印结果 ---
print("--- 重庆 CPO 测试计划 (基于拼音精确比对) ---")
print(f"重庆 CPO 总数 (去重后): {len(chongqing_pinyin_set)} 个")
print(f"已在成都测试过: {len(already_tested_pinyin)} 个")
print(f"新的测试目标: {len(new_targets_pinyin)} 个")
print("\n" + "="*40 + "\n")

print("✅ 新的测试目标列表 (重庆):")
if not new_targets_pinyin:
    print("太棒了！所有重庆的CPO似乎都已经在成都测试过。")
else:
    # 从映射中找回原始中文名并排序
    new_targets_orig = sorted([pinyin_to_orig_map_cq[p] for p in new_targets_pinyin])
    for cpo in new_targets_orig:
        print(f"- {cpo}")

print("\n" + "="*40 + "\n")

print("📋 已在成都测试过的 CPO 列表 (重庆也有):")
if not already_tested_pinyin:
    print("无。")
else:
    already_tested_orig = sorted([pinyin_to_orig_map_cq[p] for p in already_tested_pinyin])
    for cpo in already_tested_orig:
        print(f"- {cpo}")


In [ ]:
# /Users/QXZ6BKI/work/logbook_deployment/output/final_report_GY.csv

import pandas as pd

gy_df = pd.read_csv('/Users/QXZ6BKI/work/logbook_deployment/output/all_map_stations_GY.csv')
gy_df = gy_df[gy_df["operator_name"].isin(['贵州云谷', '户户充', '南网电动', '聚合快充', '贵州畅的'])]
gy_df[gy_df['operator_name'] == "聚合快充"]

## 整理 CPO LISST

In [ ]:
import pandas as pd

cpo_df = pd.read_excel('output/Logbook_2026.xlsx')
cpo_df.head()

In [ ]:
import pandas as pd
import os

# 定义日志文件名
logbook_file = 'output/Logbook_2026.xlsx'

try:
    # --- 1. 读取整合后的日志文件 ---
    print(f"正在读取日志文件: {logbook_file} ...")
    log_df = pd.read_excel(logbook_file)

    # --- 2. 提取、去重、排序 CPO 名称 ---
    # 使用 .dropna() 清除空值，.unique() 获取唯一值
    cpo_names_cn = log_df['CPO Name'].dropna().unique()
    # 排序以获得一致的顺序
    cpo_names_cn_sorted = sorted(cpo_names_cn)

    print("\\n" + "="*40)
    print("从日志中提取到的所有不重复 CPO 名称:")
    for name in cpo_names_cn_sorted:
        print(f"- {name}")
    print(f"\\n总计: {len(cpo_names_cn_sorted)} 个")
    print("="*40 + "\\n")

    # --- 3. 创建 CPO 主列表文件 ---
    master_list_file = 'cpo_master_list.csv'
    
    # 创建一个 DataFrame
    master_df = pd.DataFrame({
        'cpo_name_cn': cpo_names_cn_sorted,
        'cpo_name_en': ''  # 创建一个空的英文名列，待填充
    })

    # 保存为 CSV 文件
    master_df.to_csv(master_list_file, index=False, encoding='utf-8-sig')
    
    print(f"✅ 成功创建 CPO 主列表文件: '{master_list_file}'")
    print("下一步，请打开这个 CSV 文件，为每一个中文名填写对应的英文名。")

except FileNotFoundError:
    print(f"❌ 错误：找不到文件 '{logbook_file}'。")
    print("请确认文件名是否正确，以及文件是否位于项目根目录下。")
except KeyError:
    print(f"❌ 错误：在文件 '{logbook_file}' 中找不到名为 'CPO Name' 的列。")
    print("请检查列名是否正确。")
except Exception as e:
    print(f"处理文件时发生未知错误: {e}")


In [ ]:
!pip install opencc-python-reimplemented pypinyin

In [ ]:

# 安装必要的库
import pandas as pd
import opencc
from pypinyin import pinyin, Style
import os

# --- 准备工作 ---

# 1. 初始化简繁转换器
cc_s2t = opencc.OpenCC('s2t')  # 简体到繁体
cc_t2s = opencc.OpenCC('t2s')  # 繁体到简体

# 2. 定义拼音转换函数
def to_pinyin_flat(text):
    """将汉字字符串转换为无声调的、以下划线连接的拼音"""
    if not isinstance(text, str):
        return ""
    # 先统一转为简体再生成拼音，确保一致性
    simplified_text = cc_t2s.convert(text)
    pinyin_list = pinyin(simplified_text, style=Style.NORMAL)
    # 将 [['zhong'], ['guo']] 转换为 'zhong_guo'
    return '_'.join([item[0] for item in pinyin_list if item[0]])

# --- 数据处理 ---

# 定义日志文件名
logbook_file = 'output/Logbook_2026.xlsx'

try:
    # 1. 读取整合后的日志文件
    print(f"正在读取日志文件: {logbook_file} ...")
    log_df = pd.read_excel(logbook_file)

    # 2. 提取、去重 CPO 名称
    cpo_names_orig = log_df['CPO Name'].dropna().unique()

    # 3. 创建包含三列数据的列表
    master_list_data = []
    for name in cpo_names_orig:
        simplified_name = cc_t2s.convert(name)
        traditional_name = cc_s2t.convert(name)
        pinyin_name = to_pinyin_flat(simplified_name)
        master_list_data.append({
            'cpo_name_cn_t': traditional_name,
            'cpo_name_cn_s': simplified_name,
            'cpo_name_en': pinyin_name
        })
    
    # 4. 创建 DataFrame 并去重
    master_df = pd.DataFrame(master_list_data)
    # 基于拼音列去重，因为不同写法可能指向同一个 CPO
    master_df.drop_duplicates(subset=['cpo_name_en'], inplace=True)
    # 按拼音排序
    master_df.sort_values(by='cpo_name_en', inplace=True)
    master_df.reset_index(drop=True, inplace=True)

    # --- 结果展示与保存 ---
    
    print("\n" + "="*50)
    print("CPO 主列表预览:")
    # 使用 display() 以获得更好的表格格式
    display(master_df)
    print("="*50 + "\n")

    # 5. 保存为 CSV 文件
    master_list_file = 'cpo_master_list.csv'
    master_df.to_csv(master_list_file, index=False, encoding='utf-8-sig')
    
    print(f"✅ 成功创建并写入 CPO 主列表文件: '{master_list_file}'")
    print("您可以检查文件内容，如果需要，可以手动调整英文名。")

except FileNotFoundError:
    print(f"❌ 错误：找不到文件 '{logbook_file}'。")
    print("请确认文件名是否正确，以及文件是否位于项目根目录下。")
except KeyError:
    print(f"❌ 错误：在文件 '{logbook_file}' 中找不到名为 'CPO Name' 的列。")
    print("请检查列名是否正确。")
except Exception as e:
    print(f"处理文件时发生未知错误: {e}")


## 城市重要性 PTPI

In [ ]:
import pandas as pd

car_counts_file = 'car_counts_city.csv'
national_charging_data_file = 'national_charge_station.csv'

car_counts_df = pd.read_csv(car_counts_file)
chargeing_station_df = pd.read_csv(national_charging_data_file)

In [ ]:

# 使用您提供的更准确的新能源车保有量数据
new_car_counts_data = {
    'city': ['成都市', '重庆市', '贵阳市'],
    'unique_vehicle_count': [1080000, 952000, 200000]
}
car_counts_target_city_df = pd.DataFrame(new_car_counts_data)

print("--- 使用更新后的新能源车保有量数据 ---")
display(car_counts_target_city_df)


In [ ]:
chargeing_station_target_city_df = chargeing_station_df[chargeing_station_df["city"].isin(city_list)].reset_index(drop=True)
chargeing_station_counts_df = chargeing_station_target_city_df["city"].value_counts().to_frame().reset_index().rename(columns={"count": "station_counts"})
chargeing_station_counts_df

In [ ]:
cpo_counts_df = chargeing_station_target_city_df.groupby("city")["operator_name"].nunique().to_frame().reset_index().rename(columns={"operator_name": "CPO_counts"})
cpo_counts_df

In [ ]:

# --- 步骤 4: 合并数据 ---
# 使用 pd.merge 来合并数据
merged_df = pd.merge(car_counts_target_city_df, chargeing_station_counts_df, on='city')
merged_df = pd.merge(merged_df, cpo_counts_df, on='city')

# --- 步骤 5: 定义权重并计算 PTPI ---
w1 = 0.3  # 车辆数权重
w2 = 0.4  # CPO数权重
w3 = 0.3  # 充电站数权重

merged_df['PTPI'] = (merged_df['unique_vehicle_count'] * w1 + 
                     merged_df['CPO_counts'] * w2 + 
                     merged_df['station_counts'] * w3)

# --- 步骤 6: 计算全国总数以供对比 ---
# 使用您提供的更准确的全国总数
total_vehicle_count = 36890000 
total_station_count = chargeing_station_df.shape[0]
# total_vehicle_count = car_counts_df['unique_vehicle_count'].sum()
total_station_count = chargeing_station_df.shape[0]
total_cpo_count = chargeing_station_df['operator_name'].nunique()

# 将城市数据与全国数据合并，以便于比较
merged_df['vehicle_percentage'] = (merged_df['unique_vehicle_count'] / total_vehicle_count) * 100
merged_df['station_percentage'] = (merged_df['station_counts'] / total_station_count) * 100
merged_df['cpo_percentage'] = (merged_df['CPO_counts'] / total_cpo_count) * 100


# --- 步骤 7: 显示最终结果 ---
# 重新排列列的顺序以便查看
final_ptpi_df = merged_df[[
    'city', 
    'PTPI',
    'unique_vehicle_count', 
    'vehicle_percentage',
    'station_counts',
    'station_percentage',
    'CPO_counts',
    'cpo_percentage'
]].sort_values(by='PTPI', ascending=False).reset_index(drop=True)

print("--- 城市选择重要系数 (PTPI) 计算结果 ---")
print(f"权重: 车辆数={w1}, CPO数={w2}, 充电站数={w3}\\n")

# 格式化输出，使百分比更易读
final_ptpi_df['vehicle_percentage'] = final_ptpi_df['vehicle_percentage'].map('{:.2f}%'.format)
final_ptpi_df['station_percentage'] = final_ptpi_df['station_percentage'].map('{:.2f}%'.format)
final_ptpi_df['cpo_percentage'] = final_ptpi_df['cpo_percentage'].map('{:.2f}%'.format)


display(final_ptpi_df)

print("\\n--- 全国数据总览 ---")
print(f"全国总车辆数: {total_vehicle_count}")
print(f"全国总充电站数: {total_station_count}")
print(f"全国总CPO数: {total_cpo_count}")


In [ ]:

# --- 步骤 8: 计算三个城市合并后的 CPO 覆盖率 ---

# 1. 从之前筛选出的目标城市充电站数据中，获取每个城市的 CPO 集合
cpo_cd_set = set(chargeing_station_target_city_df[chargeing_station_target_city_df['city'] == '成都市']['operator_name'].unique())
cpo_cq_set = set(chargeing_station_target_city_df[chargeing_station_target_city_df['city'] == '重庆市']['operator_name'].unique())
cpo_gy_set = set(chargeing_station_target_city_df[chargeing_station_target_city_df['city'] == '贵阳市']['operator_name'].unique())

# 2. 将三个城市的 CPO 集合合并，得到一个总的、不重复的 CPO 集合
combined_cpo_set = cpo_cd_set.union(cpo_cq_set).union(cpo_gy_set)
combined_cpo_count = len(combined_cpo_set)

# 3. 计算覆盖率 (total_cpo_count 来自上一个单元格的计算结果)
try:
    coverage_percentage = (combined_cpo_count / total_cpo_count) * 100
    
    print("--- 成都、重庆、贵阳三城 CPO 覆盖率分析 ---")
    print(f"成都、重庆、贵阳三地合计拥有不重复的 CPO 数量: {combined_cpo_count} 个")
    print(f"全国总 CPO 数量: {total_cpo_count} 个")
    print(f"三城合计覆盖了全国 CPO 的: {coverage_percentage:.2f}%")
    
except NameError:
    print("❌ 错误: 'total_cpo_count' 未定义。请先确保上一个计算 PTPI 的单元格已成功运行。")



## 城市熱力圖

## 城市充电站热力图与已访问站点标注

In [ ]:

import pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Geo
from pyecharts.globals import ChartType

def create_city_heatmap_with_visited_stations(city_name, all_stations_df, visited_stations_df, amap_api_key):
    """
    创建城市充电站热力图，并在图上标注已访问的站点。

    :param city_name: 城市名称 (例如: "成都")
    :param all_stations_df: 包含所有充电站的 DataFrame，需要有 'longitude' 和 'latitude' 列。
    :param visited_stations_df: 包含已访问充电站的 DataFrame，需要有 'longitude' 和 'latitude' 列。
    :param amap_api_key: 您的高德地图 API Key.
    :return: pyecharts.charts.Geo 对象
    """
    
    # 准备热力图数据：[[lng, lat, 1], [lng, lat, 1], ...]
    heatmap_data = [[row['longitude'], row['latitude'], 1] for index, row in all_stations_df.iterrows()]
    
    # 准备已访问站点数据
    visited_points = [(row['station_name'], row['longitude'], row['latitude']) for index, row in visited_stations_df.iterrows()]

    # 创建 Geo 图表实例
    geo_chart = (
        Geo()
        .add_schema(
            maptype=city_name,
            itemstyle_opts=opts.ItemStyleOpts(color="#323c48", border_color="#111"),
        )
        # 添加热力图层
        .add(
            "充电站热力图",
            heatmap_data,
            type_=ChartType.HEATMAP,
        )
        # 添加已访问站点的散点图层
        .add(
            "已访问站点",
            visited_points,
            type_=ChartType.EFFECT_SCATTER,
            symbol_size=8,
            color='blue',
        )
        .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
        .set_global_opts(
            visualmap_opts=opts.VisualMapOpts(
                is_show=True,
                min_=0,
                max_=1, # 因为我们的权重都是1，所以这里设为1
                range_color=['rgba(0,255,255,0.3)', 'rgba(0,255,0,0.6)', 'yellow', 'red']
            ),
            title_opts=opts.TitleOpts(
                title=f"{city_name} 充电站分布热力图及已访问站点",
                pos_left="center",
                pos_top="top",
                title_textstyle_opts=opts.TextStyleOpts(color="white")
            ),
            legend_opts=opts.LegendOpts(
                is_show=True,
                pos_left="left",
                orient="vertical",
                textstyle_opts=opts.TextStyleOpts(color="white")
            )
        )
    )
    
    # 注入高德地图 JS 文件
    geo_chart.js_dependencies.add("amap")
    geo_chart.add_js_funcs(f'echarts.registerMap("{city_name}", {{}}, {{}});')
    geo_chart.add_js_funcs(f'bmap.AMap.AMAP_KEY = "{amap_api_key}";')


    return geo_chart

print("✅ 热力图生成函数 'create_city_heatmap_with_visited_stations' 已准备就绪。")
print("请在下一个单元格中加载您的数据并调用此函数。")


In [ ]:
import pandas as pd
national_charging_data_file = 'national_charge_station.csv'
chargeing_station_df = pd.read_csv(national_charging_data_file)

In [ ]:

import folium
from folium.plugins import HeatMap

def create_city_heatmap_with_folium(all_stations_df, visited_stations_df):
    """
    使用 Folium 创建城市充电站热力图，并在图上标注已访问的站点。

    :param all_stations_df: 包含所有充电站的 DataFrame，需要有 'latitude', 'longitude' 列。
    :param visited_stations_df: 包含已访问充电站的 DataFrame，需要有 'latitude', 'longitude', 和 'station_name' 列。
    :return: folium.Map 对象
    """
    # --- 1. 设置地图中心和底图 ---
    # 计算地图中心点
    map_center = [all_stations_df['latitude'].mean(), all_stations_df['longitude'].mean()]
    
    # 定义高德底图
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    # 创建 Folium 地图对象
    m = folium.Map(
        location=map_center,
        zoom_start=10,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 2. 添加热力图层 ---
    # 准备热力图数据
    heat_data = [[row['latitude'], row['longitude']] for index, row in all_stations_df.iterrows()]
    HeatMap(heat_data).add_to(m)

    # --- 3. 添加已访问站点的标记 ---
    for idx, row in visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(row['station_name'], max_width=200) # 点击显示站点名称
        ).add_to(m)
        
    return m

print("✅ 使用 Folium 的热力图生成函数 'create_city_heatmap_with_folium' 已准备就绪。")
print("请在下一个单元格中加载您的数据并调用此函数。")


In [ ]:
city_ = "成都市"



all_stations_df = chargeing_station_df[chargeing_station_df["city"] == city_]
visited_stations_df = pd.read_csv("output/all_map_stations_CD.csv")

m = create_city_heatmap_with_folium(all_stations_df, visited_stations_df)
m.show_in_browser()

In [ ]:

import folium
from folium.plugins import HeatMap
import json

def create_city_heatmap_with_folium(all_stations_df, visited_stations_df, geojson_path=None):
    """
    使用 Folium 创建城市充电站热力图，标注已访问站点，并可选择性地绘制地理边界。

    :param all_stations_df: 包含所有充电站的 DataFrame，需要有 'latitude', 'longitude' 列。
    :param visited_stations_df: 包含已访问充电站的 DataFrame，需要有 'latitude', 'longitude', 和 'station_name' 列。
    :param geojson_path: (可选) GeoJSON 文件的路径，用于绘制地理边界。
    :return: folium.Map 对象
    """
    # --- 1. 设置地图中心和底图 ---
    map_center = [all_stations_df['latitude'].mean(), all_stations_df['longitude'].mean()]
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    m = folium.Map(
        location=map_center,
        zoom_start=9, # 稍微缩小一点以便看到整个边界
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 2. (可选) 添加地理边界图层 ---
    if geojson_path:
        try:
            folium.GeoJson(
                geojson_path,
                style_function=lambda feature: {
                    'fillColor': 'none', # 边界内部不填充颜色
                    'color': 'red',      # 边界线颜色
                    'weight': 2,         # 边界线宽度
                    'dashArray': '5, 5'  # 虚线样式
                }
            ).add_to(m)
        except Exception as e:
            print(f"加载 GeoJSON 文件时出错: {e}")
            print("请检查文件路径是否正确，以及文件是否为有效的 GeoJSON 格式。")


    # --- 3. 添加热力图层 ---
    heat_data = [[row['latitude'], row['longitude']] for index, row in all_stations_df.iterrows()]
    HeatMap(heat_data).add_to(m)

    # --- 4. 添加已访问站点的标记 ---
    for idx, row in visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(row['station_name'], max_width=200)
        ).add_to(m)
        
    return m

print("✅ 更新后的热力图函数 'create_city_heatmap_with_folium' 已准备就绪，增加了对 GeoJSON 边界的支持。")
print("请在下一个单元格中加载数据，并在调用函数时传入 'geojson_path' 参数。")


In [ ]:

import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap

# --- 1. 定义要处理的城市和相关信息 ---
cities_info = {
    '成都市': {
        'province_name': '成都市',
        'all_stations_file': 'national_charge_station.csv', # 假设都从这个总文件筛选
        'visited_stations_file': 'output/all_map_stations_CD.csv',
        'boundary_file': 'sichuan_boundary.geojson'
    },
    '重庆市': {
        'province_name': '重庆市',
        'all_stations_file': 'national_charge_station.csv',
        'visited_stations_file': 'output/all_map_stations_CQ.csv',
        'boundary_file': 'chongqing_boundary.geojson'
    },
    '贵阳市': {
        'province_name': '贵州省',
        'all_stations_file': 'national_charge_station.csv',
        'visited_stations_file': 'output/all_map_stations_GY.csv',
        'boundary_file': 'guizhou_boundary.geojson'
    }
}

shapefile_path = 'Province/province.shp'

# --- 2. 循环处理每个城市 ---
for city_name, info in cities_info.items():
    
    print(f"\\n{'='*20} 正在处理: {city_name} {'='*20}")
    
    # --- a. 准备地理边界文件 (如果不存在) ---
    if not pd.io.common.file_exists(info['boundary_file']):
        print(f"未找到边界文件 {info['boundary_file']}，正在从 Shapefile 创建...")
        try:
            gdf = gpd.read_file(shapefile_path, encoding='gbk')
            province_gdf = gdf[gdf['NAME'] == info['province_name']]
            if not province_gdf.empty:
                province_gdf.to_file(info['boundary_file'], driver='GeoJSON', encoding='utf-8')
                print(f"✅ 成功创建 {info['boundary_file']}")
            else:
                print(f"❌ 错误: 在 Shapefile 中找不到省份 '{info['province_name']}'")
                continue # 跳过这个城市
        except Exception as e:
            print(f"❌ 创建边界文件时出错: {e}")
            continue # 跳过这个城市
    else:
        print(f"已找到边界文件: {info['boundary_file']}")

    # --- b. 加载充电站数据 ---
    try:
        print("正在加载充电站数据...")
        # 从全国数据中筛选出当前城市的站点
        national_df = pd.read_csv(info['all_stations_file'])
        all_stations_df = national_df[national_df['city'] == city_name]
        
        # 加载已访问的站点
        visited_stations_df = pd.read_csv(info['visited_stations_file'])
        
        if all_stations_df.empty or visited_stations_df.empty:
            print(f"❌ 错误: '{city_name}' 的充电站数据为空，请检查文件名和数据内容。")
            continue
            
    except FileNotFoundError as e:
        print(f"❌ 错误: 找不到数据文件 {e.filename}。")
        continue
    except Exception as e:
        print(f"❌ 加载数据时出错: {e}")
        continue

    # --- c. 调用函数生成地图 ---
    print("正在生成地图...")
    city_map = create_city_heatmap_with_folium(
        all_stations_df=all_stations_df,
        visited_stations_df=visited_stations_df,
        geojson_path=info['boundary_file']
    )
    
    # --- d. 在 Notebook 中显示地图 ---
    print(f"--- {city_name} 地图预览 ---")
    display(city_map)

print(f"\\n{'='*50}")
print("所有城市处理完毕。")


In [ ]:

geo_df = gpd.read_file('District/district.shp', )

city_list = ["成都市", "重庆城区", "贵阳市"]
merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]
# ["万州区", "黔江区", ]
merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", ])]

# cd_geo_df = geo_df[geo_df["ct_name"] == "成都市"]
# cq_geo_df = geo_df[geo_df["ct_name"] == "重庆城区"]
# gy_geo_df = geo_df[geo_df["ct_name"] == "贵阳市"]

In [ ]:
import pandas as pd
import folium

# --- 1. 筛选和合并地理数据 ---
try:
    city_list = ["成都市", "重庆城区", "贵阳市"]
    merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]
    
    # 排除重庆市的特定区域
    merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", "开州区", "梁平区"])]
    print("✅ 成功筛选并合并了三个城市的地理数据。")

    # --- 2. 准备地图 ---
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    # --- 3. 计算合并后的中心点并创建地图 ---
    # 必须转换为 EPSG:4326 (WGS84) 坐标系，folium 才能正确处理
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    m = folium.Map(
        location=map_center,
        zoom_start=7, # 调整缩放级别以容纳所有区域
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 4. 添加合并后的边界图层 ---
    # folium.GeoJson 会自动处理 GeoDataFrame
    folium.GeoJson(
        merged_geo_df,
        style_function=lambda feature: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 2,
            'fillOpacity': 0.1,
        }
    ).add_to(m)

    # --- 5. 显示地图 ---
    print("--- 正在显示合并后的边界地图 ---")
    # 在新浏览器标签页中显示地图，效果更佳
    m.show_in_browser()
    # 如果您希望在 Notebook 中直接内联显示，请取消下面这行的注释
    # display(m)

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 GeoDataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

In [ ]:
national_df = pd.read_csv('national_charge_station.csv')
city_list = ["成都市", "重庆市", "贵阳市"]
city_stations_df = national_df[national_df["city"].isin(city_list)].reset_index(drop=True)
city_stations_df.head()

In [ ]:
# city_stations_df[city_stations_df["city"] == "重庆市"]


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# --- 1. 筛选和合并地理数据 ---
try:
	city_list = ["成都市", "重庆城区", "贵阳市"]
	merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]

	merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", "开州区", "梁平区"])]
	print("✅ 成功筛选并合并了三个城市的地理数据。")

	# --- 2. 准备地图 ---
	gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
	gaode_attribution = "Amap"

	# --- 3. 计算合并后的中心点并创建地图 ---
	center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
	map_center = [center_point.y, center_point.x]

	m = folium.Map(
			location=map_center,
			zoom_start=7,
			tiles=gaode_tiles,
			attr=gaode_attribution
	)

	# --- 4. 添加合并后的边界图层 ---
	folium.GeoJson(
			merged_geo_df,
			style_function=lambda feature: {
					'fillColor': 'blue',
					'color': 'blue',
					'weight': 2,
					'fillOpacity': 0.1,
			}
	).add_to(m)
	print("✅ 成功添加地理边界图层。")

	# --- 5. 添加热力图层 ---
	# 使用您已创建的 city_stations_df
	# 准备热力图所需的数据格式 [[lat, lng], [lat, lng], ...]
	dt_list = merged_geo_df["dt_name"].tolist()
	city_stations_df = city_stations_df[city_stations_df["district"].isin(dt_list)]
	heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]

	# 添加热力图层，并设置较小的 radius 和 blur 以获得更细的颗粒度
	HeatMap(
			heat_data, 
			radius=8, 
			blur=5
	).add_to(m)
	print("✅ 成功添加热力图层（已调整颗粒度）。")

	# --- 6. 显示地图 ---
	print("--- 正在显示带有热力图的合并边界地图 ---")
	m.show_in_browser()
# display(m)

except NameError as e:
	print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
	print(f"❌ 处理地图时发生未知错误: {e}")


In [ ]:
cd_target_stations_df = pd.read_csv("output/all_map_stations_CD.csv")
cq_target_stations_df = pd.read_csv("output/all_map_stations_CQ.csv")
gy_target_stations_df = pd.read_csv("output/all_map_stations_GY.csv")

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# --- 1. 筛选和合并地理数据 ---
try:
    city_list = ["成都市", "重庆城区", "贵阳市"]
    merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]
    
    merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", "开州区", "梁平区"])]
    print("✅ 成功筛选并合并了三个城市的地理数据。")

    # --- 2. 准备地图 ---
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    # --- 3. 计算合并后的中心点并创建地图 ---
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    m = folium.Map(
        location=map_center,
        zoom_start=7,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 4. 添加合并后的边界图层 ---
    folium.GeoJson(
        merged_geo_df,
        style_function=lambda feature: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 2,
            'fillOpacity': 0.1,
        }
    ).add_to(m)
    print("✅ 成功添加地理边界图层。")

    # --- 5. 添加热力图层 ---
    heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]
    HeatMap(heat_data, radius=8, blur=5).add_to(m)
    print("✅ 成功添加热力图层。")

    # --- 6. 新增：加载并添加已访问站点的标记 ---
    print("正在加载并标记已访问的站点...")
    cd_target_stations_df = pd.read_csv("output/all_map_stations_CD.csv")
    cq_target_stations_df = pd.read_csv("output/all_map_stations_CQ.csv")
    gy_target_stations_df = pd.read_csv("output/all_map_stations_GY.csv")
    
    all_visited_stations_df = pd.concat([cd_target_stations_df, cq_target_stations_df, gy_target_stations_df], ignore_index=True)
    
    for idx, row in all_visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3, # 标记点的大小
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(row['station_name'], max_width=200) # 点击显示站点名称
        ).add_to(m)
    print(f"✅ 成功添加 {len(all_visited_stations_df)} 个已访问站点的标记。")

    # --- 7. 显示最终地图 ---
    print("--- 正在显示最终的合并地图 ---")
    m.show_in_browser()
    # display(m)

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

In [ ]:
import pandas as pd

# 读取并展示 Logbook_2026.xlsx 的内容
logbook_file = 'output/Logbook_2026.xlsx'
logbook_df = pd.read_excel(logbook_file)
logbook_df.head()

In [ ]:
import pandas as pd
import glob
import folium
from folium.plugins import HeatMap
import os

# --- 1. 设置 API Key 并加载任务记录 ---
AMAP_API_KEY = "e04966e153cf5a14405299a75cadafdd"
geocoded_cache_file = 'geocoded_mission_stations.csv'

# 检查是否已有缓存的地理编码文件
if os.path.exists(geocoded_cache_file):
    print(f"正在从缓存文件 '{geocoded_cache_file}' 加载已编码的站点坐标...")
    actual_visited_stations_df = pd.read_csv(geocoded_cache_file)
else:
    print("未找到缓存文件，将通过 API 进行地理编码。")
    # 使用之前单元格加载的 logbook_df
    try:
        # 提取不重复的场站名称
        unique_station_names = logbook_df['Station'].dropna().unique()
        print(f"从 Logbook.xlsx 中成功提取 {len(unique_station_names)} 个不重复的场站名称。")
    except NameError:
        raise NameError("'logbook_df' 未定义。请确保您已经运行了加载 'Logbook_2026.xlsx' 的单元格。")

    # --- 2. 调用 API 获取坐标 ---
    # 传入 station_names 列表和您的 API key
    actual_visited_stations_df = geocode_stations_amap(
        station_names=unique_station_names, 
        api_key=AMAP_API_KEY
    )
    # 删除任何未能成功编码的行
    actual_visited_stations_df.dropna(subset=['longitude', 'latitude'], inplace=True)
    # 保存结果以备将来使用
    actual_visited_stations_df.to_csv(geocoded_cache_file, index=False, encoding='utf-8-sig')
    print(f"已将编码结果保存到 '{geocoded_cache_file}'。")


# --- 3. 绘制最终地图 (边界 + 热力图 + 实际访问点) ---
try:
    print("\n--- 正在绘制最终地图 ---")
    # 复用之前的地图创建逻辑
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    final_map = folium.Map(
        location=map_center,
        zoom_start=7,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # 添加边界
    folium.GeoJson(merged_geo_df, style_function=lambda x: {'fillColor': 'blue', 'color': 'blue', 'weight': 2, 'fillOpacity': 0.1}).add_to(final_map)
    print("✅ 已添加地理边界。")

    # 添加热力图
    heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]
    HeatMap(heat_data, radius=8, blur=5).add_to(final_map)
    print("✅ 已添加热力图。")

    # 添加实际访问过的站点标记
    for idx, row in actual_visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3,
            color='red', # 使用红色以示区别
            fill=True,
            fill_color='red',
            fill_opacity=0.8,
            popup=folium.Popup(row['station_name'], max_width=200)
        ).add_to(final_map)
    print(f"✅ 成功添加 {len(actual_visited_stations_df)} 个实际访问站点的标记。")

    # --- 4. 显示地图 ---
    print("--- 正在显示最终的合并地图 ---")
    # final_map.show_in_browser()
    display(final_map)

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

In [85]:
import requests
import pandas as pd
import time

def geocode_stations_amap(station_names, api_key, city=''):
    """
    使用高德地图地理编码 API (v3/geocode/geo) 将充电站名称列表转换为经纬度。

    :param station_names: 包含充电站名称的列表 (list of strings)。
    :param api_key: 您的高德 Web 服务 API 密钥。
    :param city: (可选) 指定城市名称，以提高搜索精度，例如 "成都市"。
    :return: 一个包含 'station_name', 'longitude', 'latitude' 列的 DataFrame。
    """
    geocoded_data = []
    print(f"准备开始地理编码 {len(station_names)} 个站点...")
    
    # 使用您指定的 geocode/geo 接口
    url = 'https://restapi.amap.com/v3/geocode/geo'

    for i, name in enumerate(station_names):
        params = {
            'key': api_key,
            'address': name, # 参数从 'keywords' 改为 'address'
            'city': city
        }
        
        try:
            response = requests.get(url, params=params)
            data = response.json()
            
            # 解析 geocode/geo 接口的返回结构
            if data['status'] == '1' and data['geocodes']:
                geocode = data['geocodes'][0]
                location = geocode['location']
                lng, lat = map(float, location.split(','))
                geocoded_data.append({'station_name': name, 'longitude': lng, 'latitude': lat})
                print(f"({i+1}/{len(station_names)}) ✅ 成功: {name} -> ({lng}, {lat})")
            else:
                geocoded_data.append({'station_name': name, 'longitude': None, 'latitude': None})
                print(f"({i+1}/{len(station_names)}) ❌ 失败: 未找到 '{name}' 的坐标。")

        except Exception as e:
            print(f"请求 API 时发生错误: {e}")
        
        # 加上一个小的延时，避免因请求过快而被 API 限制
        time.sleep(0.02)
        
    print("地理编码完成。")
    return pd.DataFrame(geocoded_data)

print("✅ 地理编码函数 'geocode_stations_amap' 已更新并准备就绪。")

✅ 地理编码函数 'geocode_stations_amap' 已更新并准备就绪。


In [99]:
import pandas as pd
import glob
import folium
from folium.plugins import HeatMap
import os

# --- 1. 设置 API Key 并加载任务记录 ---
AMAP_API_KEY = "e04966e153cf5a14405299a75cadafdd"
geocoded_cache_file = 'geocoded_mission_stations.csv'

# 检查是否已有缓存的地理编码文件
if os.path.exists(geocoded_cache_file):
    print(f"正在从缓存文件 '{geocoded_cache_file}' 加载已编码的站点坐标...")
    actual_visited_stations_df = pd.read_csv(geocoded_cache_file)
else:
    print("未找到缓存文件，将通过 API 进行地理编码。")
    # 使用之前单元格加载的 logbook_df
    try:
        # 提取不重复的场站名称
        unique_station_names = logbook_df['Station Name'].dropna().unique()
        print(f"从 Logbook.xlsx 中成功提取 {len(unique_station_names)} 个不重复的场站名称。")
    except NameError:
        raise NameError("'logbook_df' 未定义。请确保您已经运行了加载 'Logbook_2026.xlsx' 的单元格。")

    # --- 2. 调用 API 获取坐标 ---
    # 传入 station_names 列表和您的 API key
    actual_visited_stations_df = geocode_stations_amap(
        station_names=unique_station_names, 
        api_key=AMAP_API_KEY
    )
    # 删除任何未能成功编码的行
    actual_visited_stations_df.dropna(subset=['longitude', 'latitude'], inplace=True)
    # 保存结果以备将来使用
    actual_visited_stations_df.to_csv(geocoded_cache_file, index=False, encoding='utf-8-sig')
    print(f"已将编码结果保存到 '{geocoded_cache_file}'。")


# --- 3. 绘制最终地图 (边界 + 热力图 + 品牌化标记) ---
try:
    print("\n--- 正在绘制最终地图 ---")
    # 复用之前的地图创建逻辑
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    final_map = folium.Map(
        location=map_center,
        zoom_start=7,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # 添加边界 (颜色改为灰色)
    folium.GeoJson(
        merged_geo_df, 
        style_function=lambda x: {'fillColor': '#808080', 'color': '#808080', 'weight': 2, 'fillOpacity': 0.1}
    ).add_to(final_map)
    print("✅ 已添加地理边界 (灰色)。")

    # 添加热力图
    heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]
    HeatMap(heat_data, radius=8, blur=5).add_to(final_map)
    print("✅ 已添加热力图。")

    # 添加实际访问过的站点标记 (大头针样式)
    for idx, row in actual_visited_stations_df.iterrows():
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=folium.Popup(row['station_name'], max_width=200),
            icon=folium.Icon(color='blue', icon='car', prefix='fa') # 使用蓝色汽车图标
        ).add_to(final_map)
    print(f"✅ 成功添加 {len(actual_visited_stations_df)} 个 BMW 风格的站点标记。")

    # --- 4. 显示地图 ---
    print("--- 正在显示最终的合并地图 ---")
    final_map.show_in_browser()

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

正在从缓存文件 'geocoded_mission_stations.csv' 加载已编码的站点坐标...

--- 正在绘制最终地图 ---


/var/folders/2h/m0b45vn951zbzlgkk5953xd00000gr/T/ipykernel_83378/2784344202.py:42: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid


✅ 已添加地理边界 (灰色)。
✅ 已添加热力图。
✅ 成功添加 64 个 BMW 风格的站点标记。
--- 正在显示最终的合并地图 ---
Your map should have been opened in your browser automatically.
Press ctrl+c to return.


## CPO 統計

In [100]:
!pip install pypinyin

In [110]:
try:
    from pypinyin import pinyin, Style
except ImportError:
    print("错误：'pypinyin' 库未安装。请运行之前的单元格来安装它。")

def to_pinyin_flat(name):
    """将中文名转换为小写、以下划线分隔的无音调拼音字符串。"""
    # 检查输入是否为字符串，如果不是则返回 None
    if not isinstance(name, str):
        return None
    # 将中文转换为拼音列表，例如 [['xiǎo'], ['jú']]
    pinyin_list = pinyin(name, style=Style.NORMAL)
    # 将列表展平并用下划线连接，例如 "xiao_ju"
    return '_'.join(item[0] for item in pinyin_list)

try:
    # 检查 'logbook_df' 是否已定义
    if 'logbook_df' in locals() or 'logbook_df' in globals():
        print("正在为 'logbook_df' 的 'CPO Name' 列生成拼音...")
        # 将函数应用到 'CPO Name' 列，并将结果存储在新列 'cpo_name_pinyin' 中
        logbook_df['cpo_name_pinyin'] = logbook_df['CPO Name'].apply(to_pinyin_flat)
        
        print("已成功添加 'cpo_name_pinyin' 列。")
        
        # 显示包含新列的 DataFrame 的前几行以供检查
        display(logbook_df[['CPO Name', 'cpo_name_pinyin']].head())
    else:
        print("错误：DataFrame 'logbook_df' 未定义。请先运行加载 'Logbook_2026.xlsx' 的单元格。")

except Exception as e:
    print(f"处理过程中发生错误: {e}")
    
logbook_df.to_excel('output/Logbook_2026_with_pinyin.xlsx', index=False, )
print("已将包含拼音的新 Logbook 保存为 'output/Logbook_2026_with_pinyin.xlsx'。")		

正在为 'logbook_df' 的 'CPO Name' 列生成拼音...
已成功添加 'cpo_name_pinyin' 列。


,CPO Name,cpo_name_pinyin
0,達克雲,da_ke_yun
1,太能沖,tai_neng_chong
2,億謙,yi_qian
3,NaN,None
4,億天宸,yi_tian_chen


已将包含拼音的新 Logbook 保存为 'output/Logbook_2026_with_pinyin.xlsx'。


In [118]:
import pandas as pd

# --- 1. 定义城市简码映射 ---
city_code_map = {
    '成都': 'CD',
    '重庆': 'CQ',
    '贵阳': 'GY'
    # 如果有更多城市，可以继续在这里添加
}

try:
    # --- 2. 数据清洗与准备 ---
    # 【修正点】在处理前，先筛选掉关键信息（City, CPO Name, Station）为空的行
    required_cols = ['City', 'CPO Name', 'Station', 'Date']
    original_rows = len(logbook_df)
    logbook_df.dropna(subset=required_cols, inplace=True)
    cleaned_rows = len(logbook_df)
    if original_rows > cleaned_rows:
        print(f"已清理 {original_rows - cleaned_rows} 行关键信息为空的记录。")

    # 确保数据按时间排序
    logbook_df['datetime'] = pd.to_datetime(logbook_df['Date'])
    logbook_df.sort_values('datetime', inplace=True)
    print("已根据 'Date' 列完成时间排序。")

    # --- 3. 识别每个站点的首次出现时间 ---
    first_appearance_df = logbook_df.drop_duplicates(subset=['City', 'Station'], keep='first').copy()
    print(f"已识别出 {len(first_appearance_df)} 个不重复的充电站。")

    # --- 4. 为每个CPO内的站点分组并生成序号 ---
    first_appearance_df['cpo_station_rank'] = first_appearance_df.groupby(['City', 'cpo_name_pinyin']).cumcount() + 1
    
    # --- 5. 创建唯一的站点ID ---
    first_appearance_df['station_seq_code'] = first_appearance_df['cpo_station_rank'].apply(lambda x: f"{int(x):03d}")
    
    first_appearance_df['city_code'] = first_appearance_df['City'].map(city_code_map)
    
    # 【修正点】再次检查是否有因为城市不在映射表中而产生的NaN
    if first_appearance_df['city_code'].isnull().any():
        unmapped_cities = first_appearance_df[first_appearance_df['city_code'].isnull()]['City'].unique()
        print(f"⚠️ 警告: 以下城市在 city_code_map 中没有对应的简码，将无法生成ID: {unmapped_cities}")
        first_appearance_df.dropna(subset=['city_code'], inplace=True)

    first_appearance_df['station_unique_id'] = first_appearance_df['city_code'] + '_' + \
                                             first_appearance_df['cpo_name_pinyin'] + '_' + \
                                             first_appearance_df['station_seq_code']

    # --- 6. 将唯一ID合并回原始的logbook_df ---
    id_mapping_df = first_appearance_df[['City', 'Station', 'station_unique_id']]
    
    logbook_df = pd.merge(logbook_df, id_mapping_df, on=['City', 'Station'], how='left')
    
    print("已成功生成并添加 'station_unique_id' 列。")
    
    # --- 7. 显示结果以供检查 ---
    display(logbook_df[['datetime', 'City', 'CPO Name', 'Station', 'cpo_name_pinyin', 'station_unique_id']].head())

except KeyError as e:
    print(f"❌ 错误: DataFrame 中缺少必要的列: {e}。请确保 'logbook_df' 包含 'City', 'Station', 和 'cpo_name_pinyin'。")
except Exception as e:
    print(f"❌ 处理过程中发生未知错误: {e}")

已清理 5 行关键信息为空的记录。
已根据 'Date' 列完成时间排序。
已识别出 70 个不重复的充电站。
已成功生成并添加 'station_unique_id' 列。


,datetime,City,CPO Name,Station,cpo_name_pinyin,station_unique_id
0,2026-03-09,成都,達克雲,大充新都锦水苑充电站,da_ke_yun,CD_da_ke_yun_001
1,2026-03-09,成都,崑崙網電,特来电中国华商金融中心充电站,kun_lun_wang_dian,CD_kun_lun_wang_dian_001
2,2026-03-09,成都,太能沖,大邑县铭奇快速充电站,tai_neng_chong,CD_tai_neng_chong_001
3,2026-03-09,成都,億天宸,億天宸新能源充電站,yi_tian_chen,CD_yi_tian_chen_001
4,2026-03-09,成都,中國石油 金融中心站,中國石油,zhong_guo_shi_you_ _jin_rong_zhong_xin_zhan,CD_zhong_guo_shi_you_ _jin_rong_zhong_xin_zhan...


In [119]:
logbook_df.to_excel('output/Logbook_2026_with_station_id.xlsx', index=False, )
print("已将包含站点ID的新 Logbook 保存为 'output/Logbook_2026_with_station_id.xlsx'。")		

已将包含站点ID的新 Logbook 保存为 'output/Logbook_2026_with_station_id.xlsx'。
